# dfresearch — Experiment Analysis

Visualize autoresearch experiment results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

tsv_path = Path("results.tsv")
if not tsv_path.exists():
    print("No results.tsv found. Run experiments first.")
    print("Create it with: echo -e 'commit\\tsn34_score\\taccuracy\\tmemory_gb\\tstatus\\tdescription' > results.tsv")
    df = pd.DataFrame(columns=["commit", "sn34_score", "accuracy", "memory_gb", "status", "description"])
else:
    df = pd.read_csv(tsv_path, sep="\t")
    df["sn34_score"] = pd.to_numeric(df["sn34_score"], errors="coerce")
    df["accuracy"] = pd.to_numeric(df["accuracy"], errors="coerce")
    df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
    df["status"] = df["status"].str.strip().str.upper()

df["idx"] = range(len(df))
print(f"Loaded {len(df)} experiments")
df.head(10)

In [ ]:
# Experiment outcomes
print("Status breakdown:")
print(df["status"].value_counts())
print(f"\nKeep rate: {(df['status'] == 'KEEP').mean():.1%}")
print(f"Crash rate: {(df['status'] == 'CRASH').mean():.1%}")

In [ ]:
# All KEPT experiments
kept = df[df["status"] == "KEEP"].copy()
print(f"\n{len(kept)} experiments kept:")
kept[["commit", "sn34_score", "accuracy", "memory_gb", "description"]]

In [ ]:
# sn34_score over time
fig, ax = plt.subplots(figsize=(14, 6))

# Plot discarded (gray) and crashed (red)
discard = df[df["status"] == "DISCARD"]
crash = df[df["status"] == "CRASH"]

ax.scatter(discard["idx"], discard["sn34_score"], c="#cccccc", s=40, label="Discarded", zorder=2)
ax.scatter(crash["idx"], crash["sn34_score"], c="#ff4444", s=40, marker="x", label="Crashed", zorder=2)
ax.scatter(kept["idx"], kept["sn34_score"], c="#22cc44", s=60, label="Kept", zorder=3)

# Running best
valid = df[df["sn34_score"] > 0].copy()
if len(valid) > 0:
    valid["best_so_far"] = valid["sn34_score"].cummax()
    ax.step(valid["idx"], valid["best_so_far"], c="#1166cc", linewidth=2, label="Best so far", where="post", zorder=4)

ax.set_xlabel("Experiment #")
ax.set_ylabel("sn34_score")
ax.set_title("Autoresearch Progress")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("progress.png", dpi=150)
plt.show()

In [ ]:
# Summary statistics
if len(kept) > 0:
    baseline_score = kept.iloc[0]["sn34_score"]
    best_score = kept["sn34_score"].max()
    best_row = kept.loc[kept["sn34_score"].idxmax()]

    print(f"Baseline sn34_score:  {baseline_score:.6f}")
    print(f"Best sn34_score:      {best_score:.6f}")
    print(f"Total improvement:    {best_score - baseline_score:+.6f}")
    print(f"Best experiment:      {best_row['description']}")
    print(f"Best accuracy:        {best_row['accuracy']:.4f}")
    print(f"Best memory:          {best_row['memory_gb']:.1f} GB")
    print(f"\nExperiments per kept: {len(df) / max(len(kept), 1):.1f}")
else:
    print("No experiments kept yet.")

In [ ]:
# Top improvements (delta from previous kept)
if len(kept) > 1:
    kept = kept.copy()
    kept["delta"] = kept["sn34_score"].diff()
    top = kept.dropna(subset=["delta"]).nlargest(10, "delta")
    print("Top improvements (vs previous kept):")
    for _, row in top.iterrows():
        print(f"  {row['delta']:+.6f}  {row['description']}")

In [ ]:
# Memory usage over time
fig, ax = plt.subplots(figsize=(14, 4))
valid_mem = df[df["memory_gb"] > 0]
colors = {"KEEP": "#22cc44", "DISCARD": "#cccccc", "CRASH": "#ff4444"}
for status, group in valid_mem.groupby("status"):
    ax.scatter(group["idx"], group["memory_gb"], c=colors.get(status, "gray"), s=30, label=status)
ax.set_xlabel("Experiment #")
ax.set_ylabel("Peak VRAM (GB)")
ax.set_title("Memory Usage")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()